# 03 — KPI analysis and business recommendations

This notebook calculates the KPI scorecard, uses SQL for grouped analysis, and performs a statistical comparison of average transaction values between two branches.

In [2]:
import pandas as pd
import sqlite3
from scipy import stats
from pathlib import Path
import pandas as pd

base_url = "https://raw.githubusercontent.com/naisoisharonkarinoh/plp-capstone-a-sharon-koina/main/"

sales = pd.read_csv(
    base_url + "data/processed/sales_clean.csv",
    parse_dates=['date']
)

products = pd.read_csv(
    base_url + "data/raw/products.csv"
)

df = sales.merge(products, on='product_id')

print("Sales data:", sales.shape)
print("Products data:", products.shape)
print("Merged data:", df.shape)

display(df.head())

Sales data: (5000, 11)
Products data: (20, 4)
Merged data: (5000, 14)


,transaction_id,date,branch,customer_id,product_id,quantity,discount_rate,gross_sales_kes,discount_kes,net_sales_kes,month,product_name,category,unit_price_kes
0,T00001,2025-02-02,Nairobi,C0717,P013,1,0.05,390,19.5,370.5,2025-02,Shampoo 400ml,Personal Care,390
1,T00002,2025-10-10,Nairobi,C0539,P007,1,0.00,65,0.0,65.0,2025-10,Bread,Bakery,65
2,T00003,2025-08-27,Nairobi,C0162,P009,2,0.00,680,0.0,680.0,2025-08,Coffee 100g,Beverages,340
3,T00004,2025-06-10,Kisumu,C0930,P004,1,0.00,180,0.0,180.0,2025-06,Maize Flour 2kg,Grocery,180
4,T00005,2025-06-08,Nakuru,C0251,P013,1,0.00,390,0.0,390.0,2025-06,Shampoo 400ml,Personal Care,390


## KPI scorecard

Revenue = sum of net sales; average transaction value = revenue divided by transactions; customer count = distinct customer IDs; discount rate = total discounts divided by gross sales.

In [3]:
kpis = pd.DataFrame({'KPI':['Net revenue (KES)','Transactions','Average transaction value (KES)','Unique customers','Discount rate'], 'Value':[sales.net_sales_kes.sum(),len(sales),sales.net_sales_kes.mean(),sales.customer_id.nunique(),sales.discount_kes.sum()/sales.gross_sales_kes.sum()]})
kpis

,KPI,Value
0,Net revenue (KES),3.302320e+06
1,Transactions,5.000000e+03
2,Average transaction value (KES),6.604640e+02
3,Unique customers,9.930000e+02
4,Discount rate,1.437265e-02


In [5]:
import sqlite3

conn = sqlite3.connect(':memory:')

sales.to_sql('sales_clean', conn, index=False, if_exists='replace')

print("Database connection created successfully.")

Database connection created successfully.


## SQL analysis using pandas read_sql

In [6]:
branch_sql = '''SELECT branch, COUNT(*) AS transactions, SUM(net_sales_kes) AS revenue_kes FROM sales_clean GROUP BY branch ORDER BY revenue_kes DESC'''
pd.read_sql(branch_sql, conn)

,branch,transactions,revenue_kes
0,Nairobi,1606,1099207.20
1,Mombasa,925,651010.85
2,Nakuru,895,533231.60
3,Kisumu,865,510777.50
4,Eldoret,709,508092.65


## Statistical test — Nairobi vs Mombasa transaction values

Welch's independent-samples t-test compares mean transaction value without assuming equal variances. A p-value below 0.05 would indicate evidence of a difference at the 5% significance level.

In [8]:
a = sales.loc[sales.branch=='Nairobi','net_sales_kes']
b = sales.loc[sales.branch=='Mombasa','net_sales_kes']
t_stat, p_value = stats.ttest_ind(a, b, equal_var=False)
print(f'Nairobi mean: KES {a.mean():,.2f}')
print(f'Mombasa mean: KES {b.mean():,.2f}')
print(f't-statistic: {t_stat:.4f}; p-value: {p_value:.4f}')
print('Interpretation:', 'Evidence of a difference' if p_value < 0.05 else 'No statistically significant difference detected at 5%.')
conn.close()

Nairobi mean: KES 684.44
Mombasa mean: KES 703.80
t-statistic: -0.4135; p-value: 0.6793
Interpretation: No statistically significant difference detected at 5%.
